# Objektorientert programmering

Ole-Johan Dahl og Kristen Nygaard utviklet programmeringsspråket **Simula** ved Norsk Regnesentral på 1960-tallet. Simula regnes som det første objektorienterte språket, og Dahl og Nygaard fikk både von Neumann-medaljen og Turing-prisen – «informatikkens nobelpris» – for arbeidet.

I objektorientert programmering ser vi verden som **objekter**: personer, paraplyer, kassaapparater, kortstokker, banklån, bildefiler, farger, følelser … Programmet modellerer objektene vi trenger for å løse oppgaven. Hvilke klasser, objekter, metoder og attributter vi trenger, finner vi ut i analyse- og designfasen.

> Kjør cellene i rekkefølge med **Shift + Enter**.

## 1 Klasser, objekter, metoder og attributter

* En **klasse** er en oppskrift på hvordan objektene skal være.
* Et **objekt** (en *instans*) lages fra en klasse og kan ha metoder og attributter.
* **Metoder** er funksjoner inni klassen. De har alltid parameteren `self` først – objektets referanse til seg selv. Vi tar ikke med `self` når vi kaller metoden.
* **Attributter** er variabler inni objektet.

Objekter av samme klasse skiller seg fra hverandre ved dataene i attributtene. To objekter med likt innhold er likevel to *ulike* objekter, lagret på hvert sitt sted. Når vi sammenligner objekter, må vi vite om vi sammenligner **referansene** (`is`) eller **innholdet** (`==`).

En klasse kan lages med tre ord:

In [ ]:
class Person:
    pass

person = Person()
person.navn = 'Per'       # attributtet lages i forbifarten
print(person.navn)

Vi vil helst ikke at attributter skal kunne lages hvor som helst fra (det kan hindres med `__slots__`), men det lar vi ligge.

## 2 Konstruktøren

**Konstruktøren** kjøres automatisk når vi lager et objekt. Vi bruker den til å gi attributtene startverdier. I Python heter den `__init__`. Da har vi alle attributtene (*objektvariablene*) samlet på ett sted.

In [ ]:
class Person:
    def __init__(self, navn: str, bosted: str) -> None:
        self.navn = navn
        self.bosted = bosted

    def hils(self) -> None:
        print(f'Hei, jeg er {self.navn} og kommer fra {self.bosted}.')


personer = []
personer.append(Person('Elise', 'Bergen'))
personer.append(Person('Frida', 'Stavanger'))
personer.append(Person('Gustav', 'Oslo'))

for person in personer:
    person.hils()
    print('  ', person.navn, person.bosted)

In [ ]:
a = Person('Elise', 'Bergen')
b = Person('Elise', 'Bergen')
c = a
print(a is b, a is c)    # samme objekt?

Når vi pakker data (attributter) og funksjonalitet (metoder) sammen i klasser, får vi **modularitet**. Andre sentrale prinsipper er **arv**, **polymorfi**, **innkapsling** og **abstraksjon**.

## 3 Arv

I 1.8 så vi at vi kan **generalisere** klasser til en superklasse som subklassene arver felles metoder og attributter fra. Her har `Hund` og `Katt` superklassen `Dyr` med metoden `hils()`.

Superklassen skrives i parentes etter klassenavnet. Konstruktørene i `Hund` og `Katt` sender `navn` videre til konstruktøren i `Dyr` med `super().__init__(navn)`.

In [ ]:
class Dyr:
    def __init__(self, navn: str) -> None:
        self.navn = navn

    def hils(self) -> None:
        print(f'Jeg heter {self.navn} og er en {type(self).__name__.lower()}')


class Hund(Dyr):
    def __init__(self, navn: str) -> None:
        super().__init__(navn)


class Katt(Dyr):
    def __init__(self, navn: str) -> None:
        super().__init__(navn)


for dyr in [Hund('Hans'), Katt('Kristin')]:
    dyr.hils()

## 4 Polymorfi

**Polymorfi** betyr «mange former»: vi kan kalle metoder med samme navn, men med ulikt innhold, i objekter av ulike klasser.

Finnes metoden i både superklassen og subklassen, brukes den i subklassen – den **overstyrer** (*overrides*) metoden i superklassen. Finnes den bare i superklassen, brukes den.

In [ ]:
class Hund(Dyr):
    def hils(self) -> None:
        print('Voff! ', end='')
        super().hils()


class Katt(Dyr):
    def hils(self) -> None:
        print('Mjau! ', end='')
        super().hils()


class Sau(Dyr):
    def hils(self) -> None:
        print('Bæ! ', end='')
        super().hils()


class Mus(Dyr):          # ingen egen hils() – bruker den i Dyr
    pass


liste_med_dyr = [Hund('Hans'), Katt('Kristin'), Sau('Sigurd'), Mus('Mikke')]
for dyr in liste_med_dyr:
    dyr.hils()

Legg merke til at løkka er **uendret** selv om vi har lagt til `Sau` og `Mus`. Uten polymorfi måtte vi sjekket typen overalt der koden avhenger av dyret:

In [ ]:
for dyr in liste_med_dyr:
    klasse = type(dyr).__name__
    if klasse == 'Hund':
        print('Voff!')
    elif klasse == 'Katt':
        print('Mjau!')
    elif klasse == 'Sau':
        print('Bæ!')

Med arv gjenbruker vi kode. Med polymorfi bestemmer hver klasse selv hvilken form koden skal ha.

På grunn av [*duck typing*](https://en.wikipedia.org/wiki/Duck_typing) – «går den som en and og kvekker som en and, er det en and» – trenger ikke Python at metoden finnes i superklassen. Så lenge objektet har metoden, går det bra.

## 5 Tilgang og innkapsling

I Java og JavaScript styres tilgangen med **synlighetsmodifikatorer**:

| Modifikator | Synlig for |
|---|---|
| (standard) | pakken |
| `public` | alle |
| `private` | bare klassen |
| `protected` | pakken og subklasser |

Disse finnes ikke i Python. Alle attributter kan i utgangspunktet leses og endres utenfra med punktnotasjon.

**Innkapsling** betyr å begrense tilgangen til dataene utenfra. I andre språk gjøres attributtene `private`, og vi lager *getter*- og *setter*-metoder. I setteren kan vi **validere** verdien – for eksempel hindre en negativ alder. Python har tre alternativer:

1. **Én understrek**, `_alder`: en beskjed til andre programmerere om at attributtet er ment for internt bruk (*protected*).
2. **To understreker**, `__alder`: navnet endres (*name mangling*) til `_Person__alder`, så det oppfører seg nesten som *private*.
3. **`@property`** og **`@alder.setter`**: lager getter og setter som brukes med vanlig punktnotasjon.

In [ ]:
class Person:
    def __init__(self, alder: int) -> None:
        self.__alder = alder

person = Person(20)
print(person.__dict__)            # navnet er endret
print(person._Person__alder)      # tilgjengelig likevel …
try:
    print(person.__alder)
except AttributeError as feil:
    print('AttributeError:', feil)

In [ ]:
class Person:
    def __init__(self, alder: int) -> None:
        self.alder = alder        # bruker setteren under

    @property
    def alder(self) -> int:       # getter
        return self.__alder

    @alder.setter
    def alder(self, alder) -> None:   # setter med validering
        try:
            alder = int(alder)
        except (TypeError, ValueError):
            raise ValueError(f'alder: {alder} er ikke et heltall.')
        if alder < 0:
            raise ValueError(f'alder: {alder} må være større enn 0')
        self.__alder = alder


person = Person(20)
print(person.alder)
person.alder = 19
print(person.alder)

for ny in [-11, 'tjue']:
    try:
        person.alder = ny
    except ValueError as feil:
        print('ValueError:', feil)

Utenfra ser `alder` ut som et vanlig attributt, men det er funksjonene som kalles, så vi har kontroll på verdiene.

## 6 Abstraksjon

**Abstraksjon** betyr å skjule unødvendige detaljer, slik at programmet blir enklest mulig å bruke. Tenk på klassen som en svart boks: brukerne bryr seg om *hva* den gjør, ikke *hvordan*.

Metodene og attributtene vi gjør tilgjengelige, er **programmeringsgrensesnittet** (*API – Application Programming Interface*). Andre utviklere kan bruke det til å gjenbruke koden vår. Sammenlign med en TV: fjernkontrollen er API-et. Du bytter kanal og justerer lyden uten å bry deg om elektronikken inni.

## 7 Klassevariabler

Attributtene så langt har vært **objektvariabler** – de hører til hvert objekt. Noen ganger trenger vi variabler som hører til **klassen**.

Tenk deg at vi produserer sykler og gir hver sykkel et unikt serienummer som starter på 1. Serienummeret er en objektvariabel. Men vi må også holde styr på hvor mange sykler som er laget totalt, og det hører til klassen. En variabel som lages i klassen, men utenfor metodene, er en **klassevariabel**.

In [ ]:
class Sykkel:
    antall_sykler: int = 0                       # klassevariabel

    def __init__(self) -> None:
        Sykkel.antall_sykler += 1
        self.serienummer = Sykkel.antall_sykler  # objektvariabel

    def __str__(self) -> str:
        return f'Denne sykkelen har serienummer {self.serienummer}.'


sykler = [Sykkel() for _ in range(4)]
for sykkel in sykler:
    print(sykkel)

print(f'\nAntall sykler: {Sykkel.antall_sykler}')

### `__str__`

`print(objekt)` kaller `str(objekt)`, som igjen kaller `__str__`. Uten egen `__str__` får vi noe lite nyttig som `<__main__.Sykkel object at 0x…>`. Lager vi vår egen, bestemmer vi hva som skrives ut.

## 8 Klassemetoder

**Klassemetoder** har tilgang til klassevariablene, men ikke til objektvariablene. De brukes blant annet til å gjøre det enklere å lage objekter, for eksempel fra data i en fil.

Vi lager en klassemetode med dekoratoren `@classmethod`. Den første parameteren heter `cls` (klassen), slik `self` er objektet. Vi kaller den med klassenavnet foran.

In [ ]:
%%writefile dyr.csv
Hans,Hund
Hege,Hund
Kristin,Katt
Kåre,Katt
Knut,Katt

In [ ]:
import csv

class Dyr:
    def __init__(self, navn: str) -> None:
        self.navn = navn

    def hils(self) -> None:
        print(f'Jeg heter {self.navn} og er en {type(self).__name__.lower()}')

    @classmethod
    def fra_csv(cls, datafil: str) -> list:
        typer = {'Hund': Hund, 'Katt': Katt}
        liste_med_dyr = []
        with open(datafil, encoding='utf-8') as f:
            for navn, art in csv.reader(f):
                liste_med_dyr.append(typer[art](navn))
        return liste_med_dyr


class Hund(Dyr):
    def hils(self) -> None:
        print('Voff! ', end='')
        super().hils()


class Katt(Dyr):
    def hils(self) -> None:
        print('Mjau! ', end='')
        super().hils()


for dyr in Dyr.fra_csv('dyr.csv'):
    dyr.hils()

## 9 Modellering

I klassediagrammer viser vi synligheten med et tegn foran navnet, og **klassevariabler og klassemetoder er understreket** (`{static}` i PlantUML):

| Python | Synlighet | Tegn i UML |
|---|---|---|
| `navn` | public | `+` |
| `_navn` | protected | `#` |
| `__navn` | private | `-` |
| bare `@property` (ingen setter) | kan bare leses | `{readOnly}` |

```
@startuml Synlighet
class Klassenavn {
  {static} - klassevariabel
  {static} # klassevariabel
  {static} + klassevariabel
  - objektvariabel
  # objektvariabel
  + objektvariabel
  + objektvariabel {readOnly}
  {static} + klassemetode()
  - objektmetode()
  + objektmetode()
}
@enduml
```

Klassediagrammet for sykkelen:

```mermaid
classDiagram
    class Sykkel {
        +int antall_sykler$
        +int serienummer
        +__str__() str
    }
```

## 10 Oppgaver

### Oppgave 1 – Bankkonto
Lag et program for bankkontoer. En konto har **eier**, **kontonummer** og **saldo** (beløpet på kontoen til enhver tid). Saldoen er 0 når kontoen opprettes. Det skal gå an å sette inn og ta ut penger, og programmet skal kvittere med beløpet og ny saldo. Det skal ikke gå an å ta ut mer enn det som står på kontoen – da skal det komme en feilmelding.

a) Lag et klassediagram.

b) Lag programmet. Eksempel på bruk og utskrift:

```python
konto = Konto('Hege Hermansen', '1020.30.45678')
konto.sett_inn(1000)
konto.sett_inn(2000)
konto.ta_ut(1500)
konto.ta_ut(2500)
```
```
Nyopprettet konto:
    Eier       : Hege Hermansen
    Kontonummer: 1020.30.45678
    Saldo      : 0

Du satte inn 1000.
Ny saldo er 1000
...
Du prøver å ta ut 2500.
Ikke dekning på kontoen.
Saldoen er 1500
```

Støtt deg gjerne på artikkelen *8 Tips for Object-Oriented Programming in Python* (søk på nettet) i resten av oppgaven.

c) Lag attributtene i konstruktøren hvis du ikke har gjort det, og lag en egen `__str__` slik at `print(konto)` gir en brukervennlig utskrift.

d) Beskytt `saldo` så det blir *private*, kan leses med punktnotasjon, men **ikke** endres: `konto.saldo = 10000` skal gi `AttributeError`.

e) Lag en privat klassevariabel `__MIN_SALDO = -5000`, som angir hvor mye kontoen kan overtrekkes.

f) Lag en klassemetode `fra_csv` som leser eier og kontonummer fra en fil og oppretter kontoen.

g) Oppdater klassediagrammet så det skiller mellom klasse- og objektattributter og -metoder, og viser synligheten (public, protected, private).

In [ ]:
%%writefile konto.csv
Hege Hermansen,1020.30.45678

In [ ]:
# Oppgave 1
class Konto:
    ...